In [ ]:
import os
from pathlib import Path
from collections import Counter
import cv2
import matplotlib.pyplot as plt
import yaml

# High resolution plots for thesis
plt.rcParams["figure.dpi"] = 300
plt.rcParams["font.size"] = 11

# Road Damage Detection classes
CLASSES = {0: "Pothole", 1: "Crack", 2: "Manhole"}
CLASS_COLORS = {0: (255, 0, 0), 1: (0, 0, 255), 2: (0, 255, 0)}  # RGB colors

In [ ]:
# Update these paths to match your local or Kaggle environment
RAW_DATASET_DIR = "/kaggle/input/road-damage-dataset/data"
OUTPUT_DIR = "data/road_damage_yolo"

# Run preprocess.py (70% train, 20% val, 10% test)
!python preprocess.py \
    --source {RAW_DATASET_DIR} \
    --dest {OUTPUT_DIR} \
    --train-ratio 0.7 \
    --val-ratio 0.2

In [ ]:
labels_dir = Path(OUTPUT_DIR) / "train" / "labels"
counts = Counter()

for lbl_file in labels_dir.glob("*.txt"):
    with open(lbl_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                counts[int(parts[0])] += 1

# Prepare data for plotting
names = [CLASSES[k] for k in sorted(counts.keys())]
values = [counts[k] for k in sorted(counts.keys())]
total = sum(values)

plt.figure(figsize=(7, 4))
bars = plt.bar(names, values, color=["#e74c3c", "#3498db", "#2ecc71"])

# Annotate counts and percentages on top of bars
for bar in bars:
    yval = bar.get_height()
    pct = (yval / total) * 100
    plt.text(bar.get_x() + bar.get_width() / 2.0, yval + 15, f"{yval}\n({pct:.1f}%)", ha="center", va="bottom")

plt.title("Road Damage Dataset - Training Class Distribution")
plt.xlabel("Damage Category")
plt.ylabel("Number of Bounding Boxes")
plt.ylim(0, max(values) * 1.2)
plt.grid(axis="y", linestyle="--", alpha=0.6)
plt.savefig("thesis_class_distribution.png", bbox_inches="tight")
plt.show()

In [ ]:
yaml_file = Path(OUTPUT_DIR) / "data.yaml"
with open(yaml_file, "r") as f:
    content = yaml.safe_load(f)

print("📄 data.yaml content:")
print(yaml.dump(content, default_flow_style=False))